# Silver Layer — ICU Ingredient Events (`ingredientevents`)

**Source:** `<source_schema>.ingredientevents` (default `cdr_mimiciv.bronze.ingredientevents`)  
**Target:** `<target_schema>.ingredientevents` (default `cdr_mimiciv.silver.ingredientevents`)

**Module:** MIMIC-IV ICU (`icu/ingredientevents`, MetaVision). Links to `patients` on `subject_id`, `admissions` on `hadm_id`, `icustays` on `stay_id` and `d_items` on `itemid`.

**Objective:** Prepare ICU ingredient-level administration records for the Silver layer while preserving patient and ICU stay identifiers, ingredient identifiers, medication and fluid information, quantities, rates, units and event timestamps.

**Grain:** One source ingredient event record: one ingredient (`itemid`) of an ICU input between `starttime` and `endtime`. Multiple ingredient records may belong to the same ICU input (`orderid` groups the ingredients of one solution; `linkorderid` links the order across rate changes). The official documentation defines no unique key, so uniqueness is **not asserted**. `statusdescription` of `Rewritten` rows are audit rows that were not delivered; they are preserved here and not filtered.

**Silver rules:** Preserve every Bronze record, source column and clinical value (pass-through of Bronze types). Do not delete, filter, deduplicate, impute, aggregate or automatically correct clinical information.

**Data quality:** Inspect missing values, exact duplicates, identifiers, ingredient amounts, rates, timestamps and Bronze–Silver record counts. Document anomalies without modifying or removing records.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped. Hardcoded profiled counts are asserted only when widget `enforce_profiled_counts` is `true`.

**Data quality findings:** Bronze and Silver contain 14,253,480
records with matching schemas. Fifteen records have end timestamps
earlier than start timestamps; these source anomalies are preserved
without correction. All checked timestamps and clinical quantities
are parseable. Duplicate and identifier NULL findings are documented
after validation (the reviewed code expects 0 exact duplicate groups and 0 NULL identifiers).

**Cost note:** about 14M rows. Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

# Configure the ingredientevents source and target
SOURCE_TABLE = f"{BRONZE}.ingredientevents"
TARGET_TABLE = f"{SILVER}.ingredientevents"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

Source: cdr_mimiciv.bronze.ingredientevents
Target: cdr_mimiciv.silver.ingredientevents
Enforce profiled counts: True


In [0]:
# Load all ingredient events without modifying records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()

bronze_count = bronze_df.count()

print("Bronze rows:", bronze_count)
display(bronze_df.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- amountuom: string (nullable = true)
 |-- rate: double (nullable = true)
 |-- rateuom: string (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- linkorderid: integer (nullable = true)
 |-- statusdescription: string (nullable = true)
 |-- originalamount: integer (nullable = true)
 |-- originalrate: double (nullable = true)

Bronze rows: 14253480


subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,rate,rateuom,orderid,linkorderid,statusdescription,originalamount,originalrate
10000032,29079034,39553978,18704,2180-07-23T17:00:00.000Z,2180-07-23T17:01:00.000Z,2180-07-23T18:56:00.000Z,220490,200.0,mL,null,null,3782233,3782233,FinishedRunning,0,200.0
10000032,29079034,39553978,18704,2180-07-23T17:00:00.000Z,2180-07-23T17:01:00.000Z,2180-07-23T18:56:00.000Z,227075,200.0,mL,null,null,3782233,3782233,FinishedRunning,0,200.0
10000032,29079034,39553978,18704,2180-07-23T17:00:00.000Z,2180-07-23T17:30:00.000Z,2180-07-23T17:02:00.000Z,220490,49.999998807907104,mL,100.0,mL/hour,9796595,9796595,FinishedRunning,0,50.0
10000032,29079034,39553978,18704,2180-07-23T17:00:00.000Z,2180-07-23T17:30:00.000Z,2180-07-23T17:02:00.000Z,226509,49.999998807907104,mL,100.0,mL/hour,9796595,9796595,FinishedRunning,0,50.0
10000032,29079034,39553978,18704,2180-07-23T17:33:00.000Z,2180-07-23T18:03:00.000Z,2180-07-23T18:16:00.000Z,220490,49.999998807907104,mL,100.0,mL/hour,6689854,6689854,FinishedRunning,0,50.0
10000032,29079034,39553978,18704,2180-07-23T17:33:00.000Z,2180-07-23T18:03:00.000Z,2180-07-23T18:16:00.000Z,226509,49.999998807907104,mL,100.0,mL/hour,6689854,6689854,FinishedRunning,0,50.0
10000032,29079034,39553978,18704,2180-07-23T18:56:00.000Z,2180-07-23T18:57:00.000Z,2180-07-23T18:56:00.000Z,220490,100.0,mL,null,null,2314429,2314429,FinishedRunning,0,100.0
10000032,29079034,39553978,18704,2180-07-23T18:56:00.000Z,2180-07-23T18:57:00.000Z,2180-07-23T18:56:00.000Z,227075,100.0,mL,null,null,2314429,2314429,FinishedRunning,0,100.0
10000032,29079034,39553978,20925,2180-07-23T21:10:00.000Z,2180-07-23T21:11:00.000Z,2180-07-23T21:10:00.000Z,220490,100.0,mL,null,null,527133,527133,FinishedRunning,0,100.0
10000032,29079034,39553978,20925,2180-07-23T21:10:00.000Z,2180-07-23T21:11:00.000Z,2180-07-23T21:10:00.000Z,227075,100.0,mL,null,null,527133,527133,FinishedRunning,0,100.0


In [0]:
# Inspect all Bronze columns and existing data types
schema_df = spark.createDataFrame(
    [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in bronze_df.schema.fields
    ],
    ["column_name", "bronze_type", "nullable"]
)

display(schema_df)

column_name,bronze_type,nullable
subject_id,int,true
hadm_id,int,true
stay_id,int,true
caregiver_id,int,true
starttime,timestamp,true
endtime,timestamp,true
storetime,timestamp,true
itemid,int,true
amount,double,true
amountuom,string,true


In [0]:
# Preserve all source records, columns and data types
silver_df = bronze_df

silver_df.printSchema()

# Ordered expected schema (pass-through of Bronze)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- amountuom: string (nullable = true)
 |-- rate: double (nullable = true)
 |-- rateuom: string (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- linkorderid: integer (nullable = true)
 |-- statusdescription: string (nullable = true)
 |-- originalamount: integer (nullable = true)
 |-- originalrate: double (nullable = true)



In [0]:
# Single pass: missingness, identifier nulls, parseability and reversed intervals
identifier_columns = [
    c for c in ["subject_id", "hadm_id", "stay_id", "itemid", "orderid", "linkorderid"]
    if c in silver_df.columns
]
numeric_columns = [c for c in ["amount", "rate", "originalamount", "originalrate"] if c in silver_df.columns]
timestamp_columns = [c for c in ["starttime", "endtime", "storetime"] if c in silver_df.columns]

# Parse timestamps for read-only comparison; source columns are not changed
parsed_start = F.try_to_timestamp(F.col("starttime"))
parsed_end = F.try_to_timestamp(F.col("endtime"))
reversed_cond = (
    parsed_start.isNotNull() & parsed_end.isNotNull() & (parsed_end < parsed_start)
)

profile = silver_df.agg(
    F.count("*").alias("total_records"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in silver_df.columns],
    *[
        F.sum((F.col(c).isNotNull() & F.try_to_timestamp(F.col(c)).isNull()).cast("long")).alias(f"unparseable_{c}")
        for c in timestamp_columns
    ],
    *[
        F.sum((F.col(c).isNotNull() & F.expr(f"try_cast(`{c}` AS DOUBLE)").isNull()).cast("long")).alias(f"unparseable_{c}")
        for c in numeric_columns
    ],
    F.sum(reversed_cond.cast("long")).alias("reversed_intervals")
).first().asDict()

display(spark.createDataFrame([(k, int(v or 0)) for k, v in profile.items()], ["metric", "value"]))

silver_count = profile["total_records"]
identifier_nulls = {f"null_{c}": profile[f"null_{c}"] for c in identifier_columns}
unparseable = {k: v for k, v in profile.items() if k.startswith("unparseable_") and v}
silver_reversed = int(profile["reversed_intervals"] or 0)

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Identifier NULL counts:", identifier_nulls)
print("Unparseable non-NULL values:", unparseable)
print("Reversed intervals (preserved, review only):", silver_reversed)

metric,value
total_records,14253480
null_subject_id,0
null_hadm_id,0
null_stay_id,0
null_caregiver_id,0
null_starttime,0
null_endtime,0
null_storetime,0
null_itemid,0
null_amount,0


Bronze rows: 14253480
Silver rows: 14253480
Identifier NULL counts: {'null_subject_id': 0, 'null_hadm_id': 0, 'null_stay_id': 0, 'null_itemid': 0, 'null_orderid': 0, 'null_linkorderid': 0}
Unparseable non-NULL values: {}
Reversed intervals (preserved, review only): 15


In [0]:
# Identify exact duplicate groups without removing records (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("additional_duplicate_occurrences"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate ingredientevents groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])
print("Additional duplicate occurrences:", dup_row["additional_duplicate_occurrences"])
print("Maximum group size:", dup_row["maximum_group_size"])

Exact duplicate ingredientevents groups: 0
Records in duplicate groups: 0
Additional duplicate occurrences: 0
Maximum group size: 0


In [0]:
# Pre-publish validation: row and schema preservation always; profiled counts only when enforced
assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert bronze_df.schema == silver_df.schema, "Bronze–Silver schema mismatch"

if ENFORCE_PROFILED_COUNTS:
    assert bronze_count == 14253480, f"Row count changed from profiled 14,253,480: {bronze_count}"
    assert duplicate_count == 0, "Unexpected exact duplicate groups; review before publication"
    assert all(v == 0 for v in identifier_nulls.values()), (
        f"Identifier NULL counts require review before publication: {identifier_nulls}"
    )
    assert silver_reversed == 15, (
        "Timestamp anomaly counts changed; review before publication"
    )
    assert not unparseable, f"Unparseable timestamps or quantities: {unparseable}"

print("All pre-publish checks passed.")

All pre-publish checks passed.


In [0]:
%sql
drop table cdr_mimiciv.silver.ingredientevents;

In [0]:
# Publish all ingredientevents records (CREATE OR REPLACE), preserving source timestamp anomalies
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)
print("Source timestamp anomalies preserved:", silver_reversed)

Published: cdr_mimiciv.silver.ingredientevents
Source timestamp anomalies preserved: 15


In [0]:
# Verify published ingredientevents records and schema (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in identifier_columns],
    F.sum(reversed_cond.cast("long")).alias("reversed_intervals")
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert int(published_stats["reversed_intervals"] or 0) == silver_reversed, (
    "Published reversed-interval count differs from Bronze"
)
assert {k: v for k, v in published_stats.items() if k.startswith("null_")} == identifier_nulls, (
    "Published identifier NULL counts differ from Bronze"
)

print("Silver ingredientevents table verified successfully.")

Bronze rows: 14253480
Published rows: 14253480
Silver ingredientevents table verified successfully.


In [0]:
dbutils.notebook.exit("ingredientevents:OK")